# Bab 13 — Let’s Build a Deep Learning Framework
**Acuan:** Trask, *Grokking Deep Learning*, Bab 13 (hlm. 231–263).

Tujuan: membangun computational graph, reverse-mode automatic differentiation, dan optimizer sederhana. Buku membangun tensor/layer termasuk embedding, cross-entropy, dan recurrent layer; contoh mandiri ini membatasi mesin autograd pada skalar agar graph bercabang dan akumulasi gradien terlihat jelas. Operasi vektor dan recurrent telah dihitung manual di bab sebelumnya.

## Graph dan chain rule
Setiap hasil operasi menyimpan parent dan aturan backward. Reverse-mode menyusun node secara topologis lalu berjalan terbalik, sehingga semua jalur menuju parent berkontribusi sebelum parent diproses. Gradien harus ditambah, bukan ditimpa: pada z=x*x+x, x dipakai tiga kali. Ini bukan symbolic differentiation maupun finite difference; setiap operasi lokal menyediakan turunan analitik.

In [1]:
import math
class Value:
    def __init__(self,data,parents=()):
        self.data=float(data); self.grad=0.; self.parents=parents; self._backward=lambda:None
    def __add__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        out=Value(self.data+other.data,(self,other))
        def backward():
            self.grad+=out.grad; other.grad+=out.grad
        out._backward=backward; return out
    __radd__=__add__
    def __mul__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        out=Value(self.data*other.data,(self,other))
        def backward():
            self.grad+=other.data*out.grad; other.grad+=self.data*out.grad
        out._backward=backward; return out
    __rmul__=__mul__
    def tanh(self):
        out=Value(math.tanh(self.data),(self,))
        def backward(): self.grad+=(1-out.data**2)*out.grad
        out._backward=backward; return out
    def backward(self):
        ordered=[]; visited=set()
        def visit(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents: visit(parent)
                ordered.append(node)
        visit(self)
        for node in ordered: node.grad=0.
        self.grad=1.
        for node in reversed(ordered): node._backward()

x=Value(3); z=x*x+x; z.backward()
print('z:',z.data,'dz/dx:',x.grad)
assert z.data==12 and x.grad==7
a=Value(.4); out=(a*a+a).tanh(); out.backward()
eps=1e-6; f=lambda v:math.tanh(v*v+v)
assert abs(a.grad-(f(.4+eps)-f(.4-eps))/(2*eps))<1e-6

z: 12.0 dz/dx: 7.0


## Parameter, layer, loss, optimizer
Layer mengelola parameter dan forward; loss mengubah prediksi menjadi skalar; optimizer memperbarui parameter dari gradien. Pemisahan ini memungkinkan model berganti tanpa menulis ulang aturan chain rule. Graph baru dibangun tiap iterasi. Implementasi ini mereset gradien graph saat backward; framework besar memiliki aturan akumulasi yang lebih fleksibel dan biasanya meminta `zero_grad` secara eksplisit.

In [2]:
class Linear:
    def __init__(self): self.w=Value(.1); self.b=Value(0.)
    def __call__(self,x): return self.w*x+self.b
    def parameters(self): return [self.w,self.b]
class SGD:
    def __init__(self,parameters,lr): self.parameters=parameters; self.lr=lr
    def step(self):
        for p in self.parameters: p.data-=self.lr*p.grad
model=Linear(); optimizer=SGD(model.parameters(),.05)
data=[(-1.,-1.),(0.,1.),(1.,3.),(2.,5.)]
history=[]
for epoch in range(150):
    errors=[model(x)+(-target) for x,target in data]
    loss=sum(error*error for error in errors)*(1/len(data))
    loss.backward(); optimizer.step(); history.append(loss.data)
print('w,b,loss:',model.w.data,model.b.data,history[-1])
assert history[-1]<1e-5

w,b,loss: 1.9999983122907254 1.0000027307705615 8.217269723581838e-12


## Perluasan dan ringkasan
Tensor memerlukan broadcasting beserta reduksi gradien balik, matrix multiplication, indexing dengan akumulasi untuk indeks berulang, dan fungsi loss stabil. Recurrent layer memerlukan parameter bersama pada banyak timestep—uji graph bercabang di atas adalah dasar pentingnya. Mesin kecil ini memeriksa chain rule dan optimizer; tidak mengklaim kelengkapan, kecepatan, GPU, atau pengganti framework produksi.